# Model 1 — Flat DDM
**Outputs:**
- `models/results/flat_fit.pkl` — serialised CmdStanPy fit object
- `models/results/flat_summary.csv` — posterior summary for all parameters

In [2]:
import os
os.environ["CXX"] = "g++"

In [3]:
import pickle
import pandas as pd
from pathlib import Path
from cmdstanpy import CmdStanModel

DATA_PATH = Path("../EDA/data/dataset.csv")
STAN_FILE = Path("flat.stan")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

N_CHAINS = 4
N_WARMUP = 1000
N_SAMPLES = 1000
SEED = 42

assert DATA_PATH.exists(), f"dataset.csv not found at {DATA_PATH}"
assert STAN_FILE.exists(), f"Stan file not found at {STAN_FILE}"

def load_and_validate(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = ["choice", "RT", "difficulty_bin"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df = df.dropna(subset=required)
    df = df[(df["RT"] > 0.05)]

    print(f"Loaded: {len(df):,} trials")
    print(f"RT range: {df['RT'].min():.3f}s -- {df['RT'].max():.3f}s")
    print(f"Choice balance: {df['choice'].mean():.1%} offensive")
    return df

def build_stan_data(df: pd.DataFrame) -> dict:
    return {
        "N": len(df),
        "choice": df["choice"].astype(int).tolist(),
        "RT": df["RT"].tolist(),
        "difficulty": df["difficulty_bin"].astype(int).tolist()
    }

df = load_and_validate(DATA_PATH)
stan_data = build_stan_data(df)

model = CmdStanModel(stan_file=str(STAN_FILE))
fit = model.sample(
    data=stan_data,
    chains=N_CHAINS,
    iter_warmup=N_WARMUP,
    iter_sampling=N_SAMPLES,
    seed=SEED,
    show_progress=True
)

summary = fit.summary()
summary.to_csv(RESULTS_DIR / "flat_summary.csv")

with open(RESULTS_DIR / "flat_fit.pkl", "wb") as f:
    pickle.dump(fit, f)

summary.loc[["v_easy", "v_hard", "v_diff", "beta", "p_v_easy_gt_v_hard"]]

Loaded: 22,263 trials
RT range: 0.100s -- 2.933s
Choice balance: 60.3% offensive


17:12:53 - cmdstanpy - INFO - compiling stan file C:\Users\esabe\Documents\CSCI-4150\Cognitive-Modeling-Capstone-Project\models\flat.stan to exe file C:\Users\esabe\Documents\CSCI-4150\Cognitive-Modeling-Capstone-Project\models\flat.exe
17:13:31 - cmdstanpy - INFO - compiled model executable: C:\Users\esabe\Documents\CSCI-4150\Cognitive-Modeling-Capstone-Project\models\flat.exe
17:13:33 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]




chain 1:   0%|          | 1/2000 [00:01<39:32,  1.19s/it, (Warmup)]



chain 1:  10%|█         | 200/2000 [01:54<17:27,  1.72it/s, (Warmup)]


chain 1:  15%|█▌        | 300/2000 [02:49<16:03,  1.76it/s, (Warmup)]


chain 1:  20%|██        | 400/2000 [03:38<14:16,  1.87it/s, (Warmup)]


chain 1:  25%|██▌       | 500/2000 [04:34<13:37,  1.84it/s, (Warmup)]


chain 1:  30%|███       | 600/2000 [05:24<12:20,  1.89it/s, (Warmup)]


chain 1:  35%|███▌      | 700/2000 [06:10<10:58,  1.97it/s, (Warmup)]


17:32:50 - cmdstanpy - INFO - CmdStan done processing.


,Mean,MCSE,StdDev,5%,50%,95%,N_Eff,N_Eff/s,R_hat
v_easy,0.303869,0.000275,0.013296,0.282321,0.303720,0.325950,2335.16,0.924870,0.999841
v_hard,0.286112,0.000204,0.010726,0.268468,0.286169,0.303698,2761.30,1.093650,0.999896
v_diff,0.017757,0.000276,0.015518,-0.008013,0.017833,0.043007,3155.92,1.249940,0.999899
beta,0.452127,0.000050,0.002439,0.448072,0.452091,0.456195,2366.56,0.937305,0.999620
p_v_easy_gt_v_hard,0.874500,0.006196,0.331326,0.000000,1.000000,1.000000,2859.24,1.132440,0.999040
